# Bank Churn Prediction — Multilayer Perceptron
## 1. Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

train = pd.read_csv('data/bank_data_train.csv')
test  = pd.read_csv('data/bank_data_test.csv')

test_ids = test['ID'].values
y = train['TARGET'].values.astype(np.int8)

# Keraksiz ustunlarni o'chirish
drop_cols = train.isnull().mean()[lambda x: x > 0.80].index.tolist()
drop_cols = [c for c in drop_cols if c not in ['ID', 'TARGET']]

# TARGET va ID ni HAR IKKI fayldan o'chirish
remove_train = drop_cols + ['ID', 'TARGET']
remove_test  = [c for c in drop_cols + ['ID', 'TARGET'] if c in test.columns]

train = train.drop(columns=remove_train)
test  = test.drop(columns=remove_test)

print(f"Train: {train.shape} | Test: {test.shape}")
n_train  = len(train)
cat_cols = train.select_dtypes(include='object').columns.tolist()
combined = pd.concat([train, test], axis=0).reset_index(drop=True)
print(f"Combined: {combined.shape}")

Train: (355190, 80) | Test: (88798, 80)
Combined: (443988, 80)


In [14]:
# Categorical encoding
le = LabelEncoder()
for col in cat_cols:
    combined[col] = combined[col].fillna('Unknown')
    combined[col] = le.fit_transform(combined[col].astype(str))

# Numeric fill
for col in combined.select_dtypes(include=[np.number]).columns:
    median_val = combined[col].iloc[:n_train].median()
    if pd.isna(median_val):
        median_val = 0.0
    combined[col] = combined[col].fillna(median_val)

# Safety net
combined = combined.fillna(0)
combined = combined.astype(np.float32)

# Ajratish
X_train_full = combined.values[:n_train]
X_test_final = combined.values[n_train:]

print(f"\nNaN in X_train_full: {np.isnan(X_train_full).sum()}")
print(f"NaN in X_test_final: {np.isnan(X_test_final).sum()}")


NaN in X_train_full: 0
NaN in X_test_final: 0


In [15]:
# Train/Val split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y,
    test_size=0.20, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_val_sc   = scaler.transform(X_val).astype(np.float32)
X_test_sc  = scaler.transform(X_test_final).astype(np.float32)

print(f"Features: {X_train_sc.shape[1]}")
print(f"NaN train: {np.isnan(X_train_sc).sum()}")
print(f"NaN test:  {np.isnan(X_test_sc).sum()}")
print(f"X_train_sc: {X_train_sc.shape} | {X_train_sc.dtype}")

Features: 80
NaN train: 0
NaN test:  0
X_train_sc: (284152, 80) | float32


## 2. Baseline (Naive Classifier)

In [16]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Naive: har doim ko'p uchraydigan klassni bashorat qiladi (0)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_sc, y_train)

y_pred_dummy     = dummy.predict(X_val_sc)
y_proba_dummy    = dummy.predict_proba(X_val_sc)[:, 1]

acc_dummy = accuracy_score(y_val, y_pred_dummy)
auc_dummy = roc_auc_score(y_val, y_proba_dummy)

print("BASELINE (Most Frequent)")
print(f"Accuracy : {acc_dummy:.4f}")
print(f"AUC      : {auc_dummy:.4f}")

BASELINE (Most Frequent)
Accuracy : 0.9186
AUC      : 0.5000


## 3. Random Forest + GridSearch

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Kichik grid (tez ishlashi uchun)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth'   : [10, 20, None],
    'min_samples_leaf': [10, 20],
}

rf = RandomForestClassifier(
    class_weight='balanced',  # imbalance uchun
    random_state=42,
    n_jobs=-1
)

grid_rf = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1
)

grid_rf.fit(X_train_sc, y_train)

print(f"\nBest params: {grid_rf.best_params_}")
print(f"Best CV AUC: {grid_rf.best_score_:.4f}")

# Val set natija
best_rf = grid_rf.best_estimator_
y_proba_rf = best_rf.predict_proba(X_val_sc)[:, 1]
y_pred_rf  = best_rf.predict(X_val_sc)

acc_rf = accuracy_score(y_val, y_pred_rf)
auc_rf = roc_auc_score(y_val, y_proba_rf)

print(f"RANDOM FOREST (Val set)")
print(f"Accuracy : {acc_rf:.4f}")
print(f"AUC      : {auc_rf:.4f}")

Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best params: {'max_depth': None, 'min_samples_leaf': 10, 'n_estimators': 200}
Best CV AUC: 0.8333
RANDOM FOREST (Val set)
Accuracy : 0.8647
AUC      : 0.8328


## 4. Scikit-learn MLPClassifier

In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

param_grid_mlp = {
    'hidden_layer_sizes' : [(128, 64), (256, 128, 64)],
    'activation'         : ['relu'],
    'alpha'              : [0.0001, 0.001],
    'learning_rate_init' : [0.001],
    'max_iter'           : [100],
    'early_stopping'     : [True],
    'validation_fraction': [0.1],
}

mlp = MLPClassifier(random_state=42, verbose=False)

grid_mlp = GridSearchCV(
    mlp,
    param_grid_mlp,
    cv=3,
    scoring='roc_auc',
    verbose=2,
    n_jobs=-1
)

grid_mlp.fit(X_train_sc, y_train)

print(f"\nBest params : {grid_mlp.best_params_}")
print(f"Best CV AUC : {grid_mlp.best_score_:.4f}")

best_mlp    = grid_mlp.best_estimator_
y_proba_mlp = best_mlp.predict_proba(X_val_sc)[:, 1]
y_pred_mlp  = best_mlp.predict(X_val_sc)

acc_mlp = accuracy_score(y_val, y_pred_mlp)
auc_mlp = roc_auc_score(y_val, y_proba_mlp)

print(f"SKLEARN MLP (Val set)")
print(f"Accuracy : {acc_mlp:.4f}")
print(f"AUC      : {auc_mlp:.4f}")

Fitting 3 folds for each of 4 candidates, totalling 12 fits

Best params : {'activation': 'relu', 'alpha': 0.0001, 'early_stopping': True, 'hidden_layer_sizes': (256, 128, 64), 'learning_rate_init': 0.001, 'max_iter': 100, 'validation_fraction': 0.1}
Best CV AUC : 0.7955
SKLEARN MLP (Val set)
Accuracy : 0.9192
AUC      : 0.8009


## 5. Keras (TensorFlow)

In [19]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import roc_auc_score, accuracy_score

# Class weight (imbalance uchun)
neg, pos = np.bincount(y_train)
class_weight = {0: 1.0, 1: neg / pos}
print(f"Class weights: {class_weight}")

# Model arxitekturasi
def build_model(input_dim, dropout_rate=0.3, l2=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(256, kernel_regularizer=keras.regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),

        layers.Dense(128, kernel_regularizer=keras.regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),

        layers.Dense(64, kernel_regularizer=keras.regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),

        layers.Dense(1, activation='sigmoid')
    ])
    return model

model_keras = build_model(input_dim=X_train_sc.shape[1])

model_keras.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

model_keras.summary()

# Callbacks
cb_list = [
    callbacks.EarlyStopping(monitor='val_auc', patience=5,
                            mode='max', restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_auc', patience=3,
                                factor=0.5, mode='max', verbose=1)
]

# Training
history = model_keras.fit(
    X_train_sc, y_train,
    validation_data=(X_val_sc, y_val),
    epochs=50,
    batch_size=1024,
    class_weight=class_weight,
    callbacks=cb_list,
    verbose=1
)

# Natija
y_proba_keras = model_keras.predict(X_val_sc, batch_size=1024).flatten()
y_pred_keras  = (y_proba_keras >= 0.5).astype(int)

acc_keras = accuracy_score(y_val, y_pred_keras)
auc_keras = roc_auc_score(y_val, y_proba_keras)

print(f"KERAS (Val set)")
print(f"Accuracy : {acc_keras:.4f}")
print(f"AUC      : {auc_keras:.4f}")

Class weights: {0: 1.0, 1: 11.279688850475367}
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 256)               20736     
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                                                 
 activation (Activation)     (None, 256)               0         
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 batch_normalization_1 (Bat  (None, 128)               512       
 chNormal

## 6. TensorFlow (Low-level)

In [20]:
import tensorflow as tf
from sklearn.metrics import roc_auc_score, accuracy_score

tf.random.set_seed(42)

BATCH_SIZE = 1024
EPOCHS     = 50

# Class weight uchun sample_weight
sample_weight_train = np.where(y_train == 1, neg / pos, 1.0).astype(np.float32)

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train_sc, y_train.astype(np.float32), sample_weight_train)
).shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (X_val_sc, y_val.astype(np.float32))
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Model (tf.Module OOP uslubida)
class ChurnModel(tf.Module):
    def __init__(self, input_dim):
        super().__init__()
        init = tf.initializers.GlorotUniform()

        # Layer 1: input_dim → 256
        self.W1 = tf.Variable(init([input_dim, 256]), dtype=tf.float32)
        self.b1 = tf.Variable(tf.zeros([256]), dtype=tf.float32)
        self.bn1 = tf.keras.layers.BatchNormalization()

        # Layer 2: 256 → 128
        self.W2 = tf.Variable(init([256, 128]), dtype=tf.float32)
        self.b2 = tf.Variable(tf.zeros([128]), dtype=tf.float32)
        self.bn2 = tf.keras.layers.BatchNormalization()

        # Layer 3: 128 → 64
        self.W3 = tf.Variable(init([128, 64]), dtype=tf.float32)
        self.b3 = tf.Variable(tf.zeros([64]), dtype=tf.float32)
        self.bn3 = tf.keras.layers.BatchNormalization()

        # Output: 64 → 1
        self.W4 = tf.Variable(init([64, 1]), dtype=tf.float32)
        self.b4 = tf.Variable(tf.zeros([1]), dtype=tf.float32)

    def __call__(self, x, training=False, dropout_rate=0.3):
        # Layer 1
        x = tf.matmul(x, self.W1) + self.b1
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        if training:
            x = tf.nn.dropout(x, rate=dropout_rate)

        # Layer 2
        x = tf.matmul(x, self.W2) + self.b2
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        if training:
            x = tf.nn.dropout(x, rate=dropout_rate)

        # Layer 3
        x = tf.matmul(x, self.W3) + self.b3
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        if training:
            x = tf.nn.dropout(x, rate=dropout_rate)

        # Output
        x = tf.matmul(x, self.W4) + self.b4
        return tf.nn.sigmoid(x)

# Training loop
tf_model  = ChurnModel(input_dim=X_train_sc.shape[1])
optimizer = tf.optimizers.Adam(learning_rate=0.001)

@tf.function
def train_step(x_batch, y_batch, sw_batch):
    with tf.GradientTape() as tape:
        preds = tf_model(x_batch, training=True)
        loss  = tf.keras.losses.binary_crossentropy(
                    y_batch[:, None], preds)
        loss  = tf.reduce_mean(loss * sw_batch)
    grads = tape.gradient(loss, tape.watched_variables())
    optimizer.apply_gradients(zip(grads, tape.watched_variables()))
    return loss

best_auc   = 0.0
patience   = 5
no_improve = 0
best_weights = None

for epoch in range(EPOCHS):
    # Train
    epoch_loss = []
    for x_b, y_b, sw_b in train_ds:
        loss = train_step(x_b, y_b, sw_b)
        epoch_loss.append(float(loss))

    # Validation
    preds_val = []
    for x_b, _ in val_ds:
        p = tf_model(x_b, training=False).numpy().flatten()
        preds_val.extend(p)

    val_auc = roc_auc_score(y_val, preds_val)

    if val_auc > best_auc:
        best_auc = val_auc
        no_improve = 0
        # Weights saqlash
        best_weights = [v.numpy() for v in tape.watched_variables()] \
                       if False else None
    else:
        no_improve += 1

    print(f"Epoch {epoch+1:02d} | loss: {np.mean(epoch_loss):.4f} | val_auc: {val_auc:.4f} | best: {best_auc:.4f}")

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Final prediction
preds_final = []
for x_b, _ in val_ds:
    p = tf_model(x_b, training=False).numpy().flatten()
    preds_final.extend(p)

y_proba_tf = np.array(preds_final)
y_pred_tf  = (y_proba_tf >= 0.5).astype(int)

acc_tf = accuracy_score(y_val, y_pred_tf)
auc_tf = roc_auc_score(y_val, y_proba_tf)

print(f"TENSORFLOW (Val set)")
print(f"Accuracy : {acc_tf:.4f}")
print(f"AUC      : {auc_tf:.4f}")

C:\Users\hp\AppData\Roaming\Python\Python38\site-packages\keras\src\initializers\initializers.py:120: UserWarning: The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


Epoch 01 | loss: 1.1887 | val_auc: 0.7519 | best: 0.7519
Epoch 02 | loss: 1.0880 | val_auc: 0.7694 | best: 0.7694
Epoch 03 | loss: 1.0557 | val_auc: 0.7803 | best: 0.7803
Epoch 04 | loss: 1.0352 | val_auc: 0.7876 | best: 0.7876
Epoch 05 | loss: 1.0215 | val_auc: 0.7936 | best: 0.7936
Epoch 06 | loss: 1.0074 | val_auc: 0.7974 | best: 0.7974
Epoch 07 | loss: 0.9994 | val_auc: 0.8012 | best: 0.8012
Epoch 08 | loss: 0.9892 | val_auc: 0.8032 | best: 0.8032
Epoch 09 | loss: 0.9838 | val_auc: 0.8066 | best: 0.8066
Epoch 10 | loss: 0.9760 | val_auc: 0.8085 | best: 0.8085
Epoch 11 | loss: 0.9715 | val_auc: 0.8092 | best: 0.8092
Epoch 12 | loss: 0.9674 | val_auc: 0.8100 | best: 0.8100
Epoch 13 | loss: 0.9605 | val_auc: 0.8121 | best: 0.8121
Epoch 14 | loss: 0.9565 | val_auc: 0.8124 | best: 0.8124
Epoch 15 | loss: 0.9523 | val_auc: 0.8110 | best: 0.8124
Epoch 16 | loss: 0.9495 | val_auc: 0.8135 | best: 0.8135
Epoch 17 | loss: 0.9494 | val_auc: 0.8093 | best: 0.8135
Epoch 18 | loss: 0.9416 | val_a

## 7. NumPy FCNN (OOP, scratch'dan)

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

np.random.seed(42)

# LAYER CLASSES
class DenseLayer:
    def __init__(self, input_dim, output_dim, l2=0.001):
        self.W  = (np.random.randn(input_dim, output_dim) *
                   np.sqrt(2.0 / input_dim)).astype(np.float32)
        self.b  = np.zeros((1, output_dim), dtype=np.float32)
        self.l2 = l2
        self.x  = None
        # Adam moments
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout + self.l2 * self.W
        self.db = dout.sum(axis=0, keepdims=True)
        return dout @ self.W.T

    def adam_update(self, lr, t, beta1=0.9, beta2=0.999, eps=1e-8):
        self.mW = beta1*self.mW + (1-beta1)*self.dW
        self.vW = beta2*self.vW + (1-beta2)*self.dW**2
        mW_hat  = self.mW / (1 - beta1**t)
        vW_hat  = self.vW / (1 - beta2**t)
        self.W -= lr * mW_hat / (np.sqrt(vW_hat) + eps)

        self.mb = beta1*self.mb + (1-beta1)*self.db
        self.vb = beta2*self.vb + (1-beta2)*self.db**2
        mb_hat  = self.mb / (1 - beta1**t)
        vb_hat  = self.vb / (1 - beta2**t)
        self.b -= lr * mb_hat / (np.sqrt(vb_hat) + eps)


class ReLU:
    def __init__(self):
        self.mask = None
    def forward(self, x):
        self.mask = (x > 0).astype(np.float32)
        return x * self.mask
    def backward(self, dout):
        return dout * self.mask


class Dropout:
    def __init__(self, rate=0.3):
        self.rate = rate
        self.mask = None
    def forward(self, x, training=True):
        if training:
            self.mask = (np.random.rand(*x.shape) > self.rate).astype(np.float32)
            return x * self.mask / (1 - self.rate)
        return x
    def backward(self, dout):
        return dout * self.mask / (1 - self.rate)


class BatchNorm:
    def __init__(self, dim, eps=1e-5, momentum=0.9):
        self.gamma    = np.ones((1, dim),  dtype=np.float32)
        self.beta     = np.zeros((1, dim), dtype=np.float32)
        self.eps      = eps
        self.momentum = momentum
        self.run_mean = np.zeros((1, dim), dtype=np.float32)
        self.run_var  = np.ones((1, dim),  dtype=np.float32)
        # Adam moments
        self.mg = np.zeros_like(self.gamma)
        self.vg = np.zeros_like(self.gamma)
        self.mb = np.zeros_like(self.beta)
        self.vb_= np.zeros_like(self.beta)
        self.x_norm = self.x_c = self.std = None

    def forward(self, x, training=True):
        if training:
            mean         = x.mean(axis=0, keepdims=True)
            var          = x.var(axis=0,  keepdims=True)
            self.std     = np.sqrt(var + self.eps)
            self.x_c     = x - mean
            self.x_norm  = self.x_c / self.std
            self.run_mean = self.momentum*self.run_mean + (1-self.momentum)*mean
            self.run_var  = self.momentum*self.run_var  + (1-self.momentum)*var
        else:
            self.x_norm = (x - self.run_mean) / np.sqrt(self.run_var + self.eps)
        return self.gamma * self.x_norm + self.beta

    def backward(self, dout):
        N            = dout.shape[0]
        self.dgamma  = (dout * self.x_norm).sum(axis=0, keepdims=True)
        self.dbeta   = dout.sum(axis=0, keepdims=True)
        dx_norm      = dout * self.gamma
        dvar         = (-0.5 * dx_norm * self.x_c / self.std**3).sum(axis=0, keepdims=True)
        dmean        = (-dx_norm / self.std).sum(axis=0, keepdims=True)
        return dx_norm/self.std + 2*dvar*self.x_c/N + dmean/N

    def adam_update(self, lr, t, beta1=0.9, beta2=0.999, eps=1e-8):
        for attr, grad, m, v in [
            ('gamma', self.dgamma, self.mg, self.vg),
            ('beta',  self.dbeta,  self.mb, self.vb_)
        ]:
            m[:] = beta1*m + (1-beta1)*grad
            v[:] = beta2*v + (1-beta2)*grad**2
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            setattr(self, attr, getattr(self, attr) - lr * m_hat / (np.sqrt(v_hat) + eps))


# FCNN MODEL
class FCNN:
    def __init__(self, input_dim, hidden_dims=[256,128,64],
                 dropout_rate=0.3, l2=0.001, lr=0.001):
        self.lr = lr
        self.layers = []
        self.t = 0  # Adam step counter

        dims = [input_dim] + hidden_dims
        for i in range(len(dims)-1):
            self.layers.append(DenseLayer(dims[i], dims[i+1], l2=l2))
            self.layers.append(BatchNorm(dims[i+1]))
            self.layers.append(ReLU())
            self.layers.append(Dropout(dropout_rate))

        self.out_layer = DenseLayer(hidden_dims[-1], 1, l2=l2)

    def forward(self, x, training=True):
        for layer in self.layers:
            if isinstance(layer, (Dropout, BatchNorm)):
                x = layer.forward(x, training=training)
            else:
                x = layer.forward(x)
        x = self.out_layer.forward(x)
        return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

    def backward(self, y_true, y_pred, sample_weight):
        N    = y_true.shape[0]
        dout = (y_pred - y_true) * sample_weight / N
        dout = self.out_layer.backward(dout)
        for layer in reversed(self.layers):
            if isinstance(layer, (Dropout, BatchNorm)):
                dout = layer.backward(dout)
            else:
                dout = layer.backward(dout)

    def update(self):
        self.t += 1
        for layer in self.layers + [self.out_layer]:
            if isinstance(layer, (DenseLayer, BatchNorm)):
                layer.adam_update(self.lr, self.t)

    def predict_proba(self, x):
        return self.forward(x, training=False).flatten()


# TRAINING
INPUT_DIM  = X_train_sc.shape[1]
BATCH_SIZE = 1024
EPOCHS     = 60
PATIENCE   = 7

neg, pos   = np.bincount(y_train)
sample_w   = np.where(y_train == 1, neg/pos, 1.0).astype(np.float32)
sample_w  /= sample_w.mean()

model_np   = FCNN(input_dim=INPUT_DIM, hidden_dims=[256,128,64],
                  dropout_rate=0.3, l2=0.001, lr=0.001)

best_auc_np   = 0.0
no_improve_np = 0
n_batches     = len(X_train_sc) // BATCH_SIZE

print("NumPy FCNN (Adam) Training")
for epoch in range(EPOCHS):
    idx = np.random.permutation(len(X_train_sc))
    X_s, y_s, sw_s = X_train_sc[idx], y_train[idx], sample_w[idx]

    losses = []
    for i in range(n_batches):
        xb  = X_s [i*BATCH_SIZE:(i+1)*BATCH_SIZE]
        yb  = y_s [i*BATCH_SIZE:(i+1)*BATCH_SIZE].reshape(-1,1).astype(np.float32)
        swb = sw_s[i*BATCH_SIZE:(i+1)*BATCH_SIZE].reshape(-1,1)

        pred = model_np.forward(xb, training=True)
        loss = -np.mean(swb*(yb*np.log(pred+1e-8)+(1-yb)*np.log(1-pred+1e-8)))
        losses.append(loss)

        model_np.backward(yb, pred, swb)
        model_np.update()

    # Validation
    preds = np.concatenate([
        model_np.predict_proba(X_val_sc[i:i+BATCH_SIZE])
        for i in range(0, len(X_val_sc), BATCH_SIZE)
    ])
    val_auc = roc_auc_score(y_val, preds)

    if val_auc > best_auc_np:
        best_auc_np   = val_auc
        no_improve_np = 0
        best_preds    = preds.copy()
    else:
        no_improve_np += 1

    print(f"Epoch {epoch+1:02d} | loss: {np.mean(losses):.4f} | "
          f"val_auc: {val_auc:.4f} | best: {best_auc_np:.4f}")

    if no_improve_np >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

y_proba_np = best_preds
y_pred_np  = (y_proba_np >= 0.5).astype(int)

print(f"NumPy FCNN Adam (Val set)")
print(f"Accuracy : {accuracy_score(y_val, y_pred_np):.4f}")
print(f"AUC      : {roc_auc_score(y_val, y_proba_np):.4f}")

=== NumPy FCNN (Adam) Training ===
Epoch 01 | loss: 0.6630 | val_auc: 0.7393 | best: 0.7393
Epoch 02 | loss: 0.6047 | val_auc: 0.7586 | best: 0.7586
Epoch 03 | loss: 0.5849 | val_auc: 0.7707 | best: 0.7707
Epoch 04 | loss: 0.5713 | val_auc: 0.7775 | best: 0.7775
Epoch 05 | loss: 0.5620 | val_auc: 0.7870 | best: 0.7870
Epoch 06 | loss: 0.5559 | val_auc: 0.7891 | best: 0.7891
Epoch 07 | loss: 0.5514 | val_auc: 0.7917 | best: 0.7917
Epoch 08 | loss: 0.5475 | val_auc: 0.7939 | best: 0.7939
Epoch 09 | loss: 0.5437 | val_auc: 0.7939 | best: 0.7939
Epoch 10 | loss: 0.5429 | val_auc: 0.7993 | best: 0.7993
Epoch 11 | loss: 0.5397 | val_auc: 0.7948 | best: 0.7993
Epoch 12 | loss: 0.5371 | val_auc: 0.8028 | best: 0.8028
Epoch 13 | loss: 0.5358 | val_auc: 0.8003 | best: 0.8028
Epoch 14 | loss: 0.5342 | val_auc: 0.8015 | best: 0.8028
Epoch 15 | loss: 0.5337 | val_auc: 0.8065 | best: 0.8065
Epoch 16 | loss: 0.5319 | val_auc: 0.7970 | best: 0.8065
Epoch 17 | loss: 0.5317 | val_auc: 0.8057 | best: 0.8

## 8. Final Submission (Best Model)

In [22]:
# Keras (to'liq train data, 120 epoch)

X_full_sc = scaler.transform(X_train_full).astype(np.float32)
neg_f, pos_f = np.bincount(y)
class_weight_full = {0: 1.0, 1: neg_f / pos_f}

# Eng yaxshi arxitektura: dropout=0.2, batch=512
model_best = build_model(input_dim=X_full_sc.shape[1], dropout_rate=0.2)
model_best.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

cb_best = [
    callbacks.ReduceLROnPlateau(monitor='auc', patience=3,
                                factor=0.5, mode='max', verbose=1),
    callbacks.EarlyStopping(monitor='auc', patience=10,
                            mode='max', restore_best_weights=True)
]

history_best = model_best.fit(
    X_full_sc, y,
    epochs=120,
    batch_size=512,
    class_weight=class_weight_full,
    callbacks=cb_best,
    verbose=1
)

y_test_proba_best = model_best.predict(X_test_sc, batch_size=1024).flatten()
submission = pd.DataFrame({'ID': test_ids, 'TARGET': y_test_proba_best})
submission.to_csv('submission.csv', index=False)

auc_key = [k for k in history_best.history.keys() if 'auc' in k][0]
best_auc = max(history_best.history[auc_key])
print(f'Best Train AUC: {best_auc:.4f}')

Epoch 1/120
694/694 [==============================] - 8s 9ms/step - loss: 1.4349 - auc: 0.7171 - lr: 0.0010
Epoch 2/120
694/694 [==============================] - 6s 9ms/step - loss: 1.2209 - auc: 0.7660 - lr: 0.0010
Epoch 3/120
694/694 [==============================] - 6s 9ms/step - loss: 1.1284 - auc: 0.7780 - lr: 0.0010
Epoch 4/120
694/694 [==============================] - 6s 9ms/step - loss: 1.0823 - auc: 0.7853 - lr: 0.0010
Epoch 5/120
694/694 [==============================] - 8s 11ms/step - loss: 1.0613 - auc: 0.7888 - lr: 0.0010
Epoch 6/120
694/694 [==============================] - 7s 11ms/step - loss: 1.0517 - auc: 0.7909 - lr: 0.0010
Epoch 7/120
694/694 [==============================] - 8s 12ms/step - loss: 1.0425 - auc: 0.7940 - lr: 0.0010
Epoch 8/120
694/694 [==============================] - 7s 10ms/step - loss: 1.0386 - auc: 0.7956 - lr: 0.0010
Epoch 9/120
694/694 [==============================] - 7s 11ms/step - loss: 1.0358 - auc: 0.7965 - lr: 0.0010
Epoch 10/120
6